<a href="https://colab.research.google.com/github/miso-20/statistic_programing_2/blob/main/6_3)_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 그룹화 계산

## 1. `groupby()`

- **그룹화 계산**은 특정 변수의 값에 따라 데이터를 나누고, 각 그룹의 통계량을 계산하는 방법이다.
- 그룹화 계산은 다음 세 단계로 진행된다.
  1. **분리(Split)**: 기준 변수에 따라 데이터를 그룹으로 나눈다.
  2. **적용(Apply)**: 각 그룹에 원하는 계산을 적용한다.
  3. **결합(Combine)**: 그룹별 계산 결과를 하나로 결합한다.
- Pandas에서는 `groupby()`를 이용해 반복문 없이 그룹별 계산을 수행할 수 있다.

In [ ]:
import pandas as pd

ex_dat = pd.DataFrame( {
    "school" : ["A", "A", "A", "A", "B", "B", "B"],
    "sex" : ["M", "M", "F", "F", "M", "F", "F"],
    "score" : [100, 98, 34, 83, 56, 90, 65]
})
ex_dat

,school,sex,score
0,A,M,100
1,A,M,98
2,A,F,34
3,A,F,83
4,B,M,56
5,B,F,90
6,B,F,65


- 학교와 성별에 따라 데이터를 그룹으로 나눈 뒤, 각 그룹의 평균 점수를 계산해 보자.
- 먼저 반복문을 이용하여 그룹별 평균을 구하고 결과를 결합한다.

In [ ]:
uni_sch = ex_dat["school"].unique()
uni_sex = ex_dat["sex"].unique()

result_dat = pd.DataFrame()
for tmp_sch in uni_sch:
    for tmp_sex in uni_sex:
        tmp_ind = (ex_dat["school"] == tmp_sch) & (ex_dat["sex"] == tmp_sex)
        sub_dat = ex_dat.loc[tmp_ind, :]

        mean_dat = pd.DataFrame([[tmp_sch, tmp_sex, sub_dat["score"].mean()]],
                                columns = ['school', 'sex', 'score_mean'])

        result_dat = pd.concat([result_dat, mean_dat])

result_dat

,school,sex,score_mean
0,A,M,99.0
0,A,F,58.5
0,B,M,56.0
0,B,F,77.5


- 위 코드는 반복문으로 데이터를 그룹별로 나눈 후, `mean()`으로 평균을 계산하고 `pd.concat()`으로 결과를 결합한다.
- 같은 계산을 `groupby()`로 수행해 보자.

In [ ]:
ex_group = ex_dat.groupby(["school", "sex"])
ex_group

- `groupby()`를 적용하면 그룹 정보를 포함하는 결과가 생성된다. 이를 `groupby` 객체라고 한다.
- 먼저 데이터가 어떤 그룹으로 나뉘는지 확인해 보자.

In [ ]:
ex_group.groups

{('A', 'F'): RangeIndex(start=2, stop=4, step=1),
 ('A', 'M'): RangeIndex(start=0, stop=2, step=1),
 ('B', 'F'): RangeIndex(start=5, stop=7, step=1),
 ('B', 'M'): RangeIndex(start=4, stop=5, step=1)}

- 학교와 성별의 조합에 따라 네 개의 그룹이 생성되며, 각 그룹에 해당하는 행 인덱스를 확인할 수 있다.
- 그룹별 평균을 계산하면 각 그룹에 함수를 적용한 결과가 함께 정리된다.

In [ ]:
mean_dat = ex_dat.groupby(by = ["school", "sex"]).mean()
mean_dat

score
school sex       
A      F     58.5
       M     99.0
B      F     77.5
       M     56.0

- `mean()`을 적용하면 그룹별 평균이 계산되고, 결과가 하나로 정리된다.
- 결과의 표현 형태는 다르지만, 앞에서 반복문으로 계산한 학교별·성별 평균과 동일하다.

In [ ]:
print(result_dat.info())
print(mean_dat.info())

<class 'pandas.DataFrame'>
Index: 4 entries, 0 to 0
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   school      4 non-null      str    
 1   sex         4 non-null      str    
 2   score_mean  4 non-null      float64
dtypes: float64(1), str(2)
memory usage: 136.0 bytes
None
<class 'pandas.DataFrame'>
MultiIndex: 4 entries, ('A', 'F') to ('B', 'M')
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   score   4 non-null      float64
dtypes: float64(1)
memory usage: 169.0 bytes
None


- `info()`로 결과를 확인하면 학교와 성별이 일반 열이 아니라 행 인덱스로 사용되고 있음을 알 수 있다.
- 학교와 성별을 다시 열로 나타내려면 `reset_index()`를 이용한다.

In [ ]:
mean_dat = mean_dat.reset_index()
mean_dat

,school,sex,score
0,A,F,58.5
1,A,M,99.0
2,B,F,77.5
3,B,M,56.0


In [ ]:
mean_dat.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   school  4 non-null      str    
 1   sex     4 non-null      str    
 2   score   4 non-null      float64
dtypes: float64(1), str(2)
memory usage: 238.0 bytes


- `reset_index()`를 적용하면 학교와 성별이 일반 열로 나타나며, 반복문으로 생성한 결과와 같은 형태로 확인할 수 있다.
- 여러 수치형 열에 대해 같은 통계량을 한 번에 계산할 수도 있다.

In [ ]:
import numpy as np  # 외우지 마!

np.random.seed(1)

data = {'cate_1': np.random.choice(['A', 'B'], size=10),
        'cate_2': np.random.choice(['X', 'Y'], size=10),
        'value_1': np.random.randint(1, 100, size=10),
        'value_2': np.random.randint(1, 100, size=10)}

ex_dat = pd.DataFrame(data)
ex_dat

,cate_1,cate_2,value_1,value_2
0,B,X,19,95
1,B,Y,85,97
2,A,X,12,87
3,A,Y,29,14
4,B,Y,30,10
5,B,X,15,8
6,B,X,51,64
7,B,Y,69,62
8,B,X,88,23
9,A,X,88,58


- `np.random.seed(1)`은 난수 생성의 시작 상태를 고정하여, 코드를 다시 실행해도 같은 난수 결과를 얻도록 한다.
- `np.random.choice(['A', 'B'], size=10)`은 `A`와 `B` 중 하나를 무작위로 선택하는 작업을 10번 수행한다.
- `np.random.randint(1, 100, size=10)`은 **1 이상 100 미만**의 정수 중에서 10개를 무작위로 생성한다. 따라서 1부터 99까지의 값이 생성될 수 있다.
- `size=10`은 생성할 값의 개수를 의미한다. 이 예제에서는 네 개의 열에 각각 10개씩 값을 생성하여 10행의 `DataFrame`을 만든다.

- `cate_1`과 `cate_2`를 기준으로 그룹을 나누고, `value_1`과 `value_2`의 평균을 계산해 보자.

In [ ]:
mean_dat = ex_dat.groupby(['cate_1', 'cate_2']).mean() # 그냥 mean 쓰면 'cate_1', 'cate_2' 들 제외한 나머지 다 계산
mean_dat

value_1    value_2
cate_1 cate_2                      
A      X       50.000000  72.500000
       Y       29.000000  14.000000
B      X       43.250000  47.500000
       Y       61.333333  56.333333

- `groupby()`로 그룹을 나눈 뒤 `mean()`을 적용하면, 그룹 기준 열을 제외한 수치형 열의 평균이 계산된다.
- 이번에는 `cate_1`만 기준으로 사용하여 결과를 비교해 보자.

In [ ]:
mean_dat = ex_dat.groupby(['cate_1']).mean() # 그냥 mean 써서 cate_1제외한 나머지 다 계산 # but, cate_2 수치형 아니기 때문에 에러남
mean_dat

TypeError: dtype 'str' does not support operation 'mean'

- 위 코드에서는 `cate_1`만 그룹 기준으로 지정했으므로, `cate_2`는 그룹 기준에서 제외된다.
- `cate_2`에는 `X`, `Y`와 같은 문자열이 저장되어 있어 평균을 계산할 수 없다. 따라서 현재 Pandas 버전에서는 문자열을 숫자로 변환하려는 과정에서 `TypeError`가 발생한다.
- 평균을 구할 열을 `[["value_1", "value_2"]]`로 직접 지정하면 오류를 피할 수 있다. 아래 예제에서 확인해 보자.
- 참고로 Pandas 버전에 따라 숫자형 열만 자동으로 선택하던 경우도 있으므로, 오류 발생 여부는 버전에 따라 달라질 수 있다. # 알아야 함

In [ ]:
mean_dat = ex_dat.groupby(["cate_1"])[["value_1", "value_2"]].mean() # list로 지정하면 오류 안 남 # cate_2 안 들어가니까
                                                                     # 데이터프라임에서의 변수 선택
mean_dat

,value_1,value_2
cate_1,,
A,43.0,53.000000
B,51.0,51.285714


## 2. `agg()`를 이용한 여러 통계량 계산

- 앞에서는 각 그룹에 동일한 통계량을 적용하였다.
- `agg()`를 이용하면 열마다 서로 다른 통계량을 계산하거나, 하나의 열에 여러 통계량을 적용할 수 있다.
- 열마다 다른 함수를 지정할 때는 Dictionary 형태로 작성한다. 예를 들어 `agg({"value_1": "mean", "value_2": "sum"})`은 `value_1`의 평균과 `value_2`의 합계를 계산한다.
- 앞에서 생성한 `ex_dat`을 이용해 결과를 확인해 보자.

In [ ]:
sum_dat = ex_dat.groupby(["cate_1", "cate_2"]).agg({"value_1" : "mean", "value_2" : "sum"})
sum_dat

value_1  value_2
cate_1 cate_2                    
A      X       50.000000      145
       Y       29.000000       14
B      X       43.250000      190
       Y       61.333333      169

- `agg()`에서는 계산할 열과 함수를 직접 지정할 수 있다.
- 필요한 통계량에 맞추어 열과 함수를 선택해 보자.

In [ ]:
sum_dat = ex_dat.groupby(["cate_1"]).agg({"value_1" : "mean", "value_2" : "sum"})
sum_dat

,value_1,value_2
cate_1,,
A,43.0,159
B,51.0,359


- 하나의 열에 여러 통계량을 적용하려면 함수 이름을 리스트로 지정한다.

In [ ]:
ex_dat.groupby(["cate_1", "cate_2"]).agg({"value_1": ["sum", "count"]})

value_1      
                  sum count
cate_1 cate_2              
A      X          100     2
       Y           29     1
B      X          173     4
       Y          184     3

- 직접 정의한 함수도 `agg()`에 적용할 수 있다.
- 다음은 결측값을 제외한 평균과 표준편차, 50 이하 값의 평균을 계산하는 함수를 정의하고 적용하는 예제이다.
- dropna(): 결측값 제거   # 이런 함수 암기 안해도 됨

In [ ]:
def non_nan_mean(x):
    x = x.dropna()
    return np.mean(x)

def standard_deviation(x):
    x = x.dropna()
    return np.nanstd(x)

def mean_lower_rating(x):
    upper = []
    for val in x:
        if val > 50:
            continue
        else:
            upper.append(val)

    return np.mean(upper)     # 여기선 무조건 return 있어야 됨!!!

In [ ]:
ex_dat.groupby(["cate_1"]).agg({'value_1' : mean_lower_rating,
                               'value_2' : [standard_deviation, mean_lower_rating]})

value_1            value_2                  
       mean_lower_rating standard_deviation mean_lower_rating
cate_1                                                       
A              20.500000          30.011109         14.000000
B              21.333333          35.163757         13.666667

In [ ]:
ex_dat.groupby(["cate_1", "cate_2"]).agg({"value_1" : [non_nan_mean, standard_deviation, mean_lower_rating],
                                          "value_2" : [sum, mean_lower_rating]})

D:\anaconda3\Lib\site-packages\numpy\_core\fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
D:\anaconda3\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


value_1                                      value_2  \
              non_nan_mean standard_deviation mean_lower_rating     sum   
cate_1 cate_2                                                             
A      X         50.000000          38.000000              12.0     145   
       Y         29.000000           0.000000              29.0      14   
B      X         43.250000          29.363029              17.0     190   
       Y         61.333333          23.098822              30.0     169   

                                 
              mean_lower_rating  
cate_1 cate_2                    
A      X                    NaN  
       Y                   14.0  
B      X                   15.5  
       Y                   10.0

시험엔 안나오지만 중요한 것
-> 코드 돌아가도 warning 뜨면 반드시 이유 알아보기

=> 바로 위 코드에서 warning 나오는 이유: invalid value encountered in scalar divide (데이터가 존재하지 않기 때문에 오류남)